# 12. PITWALL Replay Engine & Strategy Replay Pipeline

This notebook runs the end-to-end replay engine for session 9558 (Silverstone 2024, 52 laps).
It demonstrates:
1. Loading validated session data and detecting clean laps, pit in/out laps, and finishers.
2. Fitting the lap fixed-effects pace model to isolate compound and degradation effects without weather confounding.
3. Measuring empirical pit loss across historical green-flag stops.
4. Generating procedural multi-stop strategy candidates (k=1 and k=2 stops).
5. Running the acceptance test suite (Tests A, B, C, D) with multi-stop baseline evaluation.
6. Re-running the original motivating scenarios: Driver 63 (Russell) vs Driver 44 (Hamilton) at lap 26.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pitwall.data import load_race, get_finishers, get_driver_laps
from pitwall.pace_model import PaceModel
from pitwall.pit_loss import measure_pit_loss
from pitwall.strategy_generator import generate_multi_stop_candidates
from pitwall.engine import replay, replay_actual, Stay, Pit, Strategy
from pitwall.validation import run_plumbing_test, run_identity_test, run_sanity_bound_test, run_support_test

# 1. Load Session 9558 Data
df = load_race(9558)
finishers = get_finishers(df)
print(f'Loaded session 9558: {len(df)} lap records across {df["driver_number"].nunique()} drivers.')
print(f'Finishers ({len(finishers)}): {finishers}')
print(f'Clean laps identified: {df["is_clean_lap"].sum()}')

## 2. Fit Lap Fixed-Effects Pace Model

The model formula is:
$$\text{lap\_duration} \sim C(\text{lap\_number}) + C(\text{driver\_number}) + C(\text{compound}) + C(\text{compound}):\text{tyre\_age}$$
The per-lap term absorbs track wetness and fuel burn, preventing compound/tyre-age confounding.

In [ ]:
model = PaceModel(reference_compound='MEDIUM').fit(df)
print(f'Model R-squared: {model.rsquared:.4f}')
rep = model.identification_report
print(f'Clean laps: {rep.n_clean_laps}, Multi-compound laps: {rep.n_multi_compound_laps}')
print('\nCompound Pair Same-Lap Identification:')
for pair, cnt in rep.pair_counts.items():
    status = '[WEAK]' if pair in rep.weakly_identified_pairs else '[OK]'
    print(f'  {status} {pair}: {cnt} laps')

# Coefficient table
coef_data = []
for c in model.coefficients:
    if 'compound' in c.param_name.lower():
        coef_data.append({'Parameter': c.param_name, 'Estimate': round(c.coef, 4), '95% CI Lower': round(c.ci_lower, 4), '95% CI Upper': round(c.ci_upper, 4), 'p-value': round(c.pvalue, 4)})
pd.DataFrame(coef_data)

## 3. Empirical Pit-Loss Measurement

Pit loss is measured directly by comparing actual in-lap + out-lap times against expected clean pace from the model.

In [ ]:
green_summary, sc_summary = measure_pit_loss(df, model)
print(f'Green-flag pit stops measured: n={green_summary.n_stops}')
print(f'Median pit loss: {green_summary.median:.2f} s (IQR: [{green_summary.iqr_low:.2f}, {green_summary.iqr_high:.2f}] s)')
print(f'Mean pit loss: {green_summary.mean:.2f} s +/- {green_summary.std:.2f} s')

# Plot pit loss distribution
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(green_summary.stops_table['pit_loss'], bins=15, color='steelblue', edgecolor='black', alpha=0.7)
ax.axvline(green_summary.median, color='red', linestyle='--', linewidth=2, label=f'Median ({green_summary.median:.2f}s)')
ax.axvline(22.0, color='orange', linestyle=':', linewidth=2, label='Old Assumption (22.0s)')
ax.set_title('Green-Flag Pit Loss Distribution (Session 9558)')
ax.set_xlabel('Pit Loss (s)')
ax.set_ylabel('Count')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Multi-Stop Strategy Candidate Generation

For driver 44 (Lewis Hamilton) at decision lap 26, procedural multi-stop candidates are generated and ranked by distance to historical baseline.

In [ ]:
cands_44 = generate_multi_stop_candidates(df, target_driver=44, decision_lap=26)
print(f'Total candidate count: {len(cands_44)}')
print('\nTop 10 candidates by baseline distance:')
display(cands_44[['strategy_id', 'pit_sequence_str', 'is_baseline', 'baseline_distance', 'num_stops']].head(10))

# Replaying the baseline candidate S001
base_row = cands_44[cands_44['is_baseline']].iloc[0]
strat_base = Strategy(pit_sequence=tuple(base_row['pit_sequence']), description=base_row['description'])
res_base = replay(df, 44, 26, strategy=strat_base, pace_model=model, pit_loss=green_summary.median)
print('\nBaseline candidate (S001) replay result:')
print(f'  Projected final position: P{res_base.summary["final_position"]} (Actual: P1)')
print(f'  Total time delta vs actual: {res_base.summary["total_time_delta_vs_actual"]:.2f} s')

## 5. Acceptance Test Suite (Tests A, B, C, D)

Test B evaluates finishers against their historical multi-stop baseline.

In [ ]:
res_a = run_plumbing_test(df)
res_b = run_identity_test(df, pace_model=model, pit_loss=green_summary.median)
res_c = run_sanity_bound_test(df, target_driver=44, decision_lap=20, pace_model=model, pit_loss=green_summary.median)
supp_ok, supp_msg = run_support_test(df, 'INTERMEDIATE', (28, 38))

test_results_summary = [
    {'Test': res_a.name, 'Status': 'PASS' if res_a.passed else 'FAIL', 'Metric': res_a.message},
    {'Test': res_b.name, 'Status': 'PASS' if res_b.passed else 'FAIL', 'Metric': res_b.message},
    {'Test': res_c.name, 'Status': 'PASS' if res_c.passed else 'FAIL', 'Metric': res_c.message},
    {'Test': 'Support Test (Test D)', 'Status': 'PASS' if supp_ok else 'FAIL', 'Metric': supp_msg}
]
display(pd.DataFrame(test_results_summary))

# Test B detailed table
tbl_b = pd.DataFrame(res_b.details['table'])
print('\nTest B Results Table (sample):')
display(tbl_b[['driver', 'decision_lap', 'pit_sequence', 'time_error_s', 'sim_final_pos', 'actual_final_pos', 'abs_pos_error']].head(15))

## 6. Gap-to-Leader vs Lap Comparison

Comparing Hamilton's actual race vs counterfactual pit strategies.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
res_stay = replay(df, 44, 26, strategy=Stay(), pace_model=model, pit_loss=green_summary.median)

ham_act = df[df['driver_number']==44]
ham_base = res_base.laps_table[res_base.laps_table['driver_number']==44]
ham_stay = res_stay.laps_table[res_stay.laps_table['driver_number']==44]

leader_times = df.groupby('lap_number')['cum_time'].min().to_dict()
act_gaps = [ham_act[ham_act['lap_number']==l]['cum_time'].iloc[0] - leader_times[l] for l in range(1, 53)]

ax.plot(range(1, 53), act_gaps, label='Actual Race (Hamilton)', color='purple', linewidth=2)
ax.plot(ham_base['lap_number'], ham_base['gap_to_leader'], label='Baseline Multi-Stop S001 (INTER 27 -> SOFT 38)', color='teal', linestyle='--', linewidth=2)
ax.plot(ham_stay['lap_number'], ham_stay['gap_to_leader'], label='Counterfactual Stay Out', color='red', linestyle=':', linewidth=2)
ax.set_title('Gap to Leader vs Lap: Hamilton Actual vs Multi-Stop Replay (Decision Lap 26)')
ax.set_xlabel('Lap Number')
ax.set_ylabel('Gap to Leader (s)')
ax.axvline(26, color='gray', linestyle='--', alpha=0.7, label='Decision Lap 26')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [1]:
from pitwall.engine import Strategy, replay

# Scenario 1: Real history (what actually happened)
real = Strategy(
    pit_sequence=((27, 'INTERMEDIATE'), (38, 'SOFT')),
    description="Real: INTER lap 27 → SOFT lap 38"
)
result_real = replay(session_df, target_driver=44, decision_lap=25, strategy=real)

# Scenario 2: What if he stayed out?
stay_out = Strategy(
    pit_sequence=((999, 'MEDIUM'),),  # never pit
    description="Counterfactual: Stay out forever"
)
result_stay = replay(session_df, target_driver=44, decision_lap=25, strategy=stay_out)

# Compare
print("Real strategy (INTER 27 → SOFT 38):")
print(f"  Final position: {result_real.summary['final_position']}")
print(f"  Final time: {result_real.summary['final_time_s']:.2f} s")

print("\nStay out forever:")
print(f"  Final position: {result_stay.summary['final_position']}")
print(f"  Final time: {result_stay.summary['final_time_s']:.2f} s")

print(f"\nDifference: {result_stay.summary['final_time_s'] - result_real.summary['final_time_s']:.2f} s")

NameError: name 'session_df' is not defined